In [80]:
from openapi_client import LotInfoShort, ProductWithId
from openapi_client import ApiClient, Configuration
from openapi_client.api import DefaultApi
import requests
import pandas as pd
from typing import List
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from datetime import datetime

In [81]:
server = "naks42.ru"
port = 17443
clientId = "Ebay.Python"
secret = "78195A38-796A-4EE0-8F2E-8F4EB3FECF34"

In [82]:
unknown_price_discount = 0.7

In [83]:
def get_access_token(url, client_id, client_secret):
    response = requests.post(
        url,
        data={"grant_type": "client_credentials"},
        auth=(client_id, client_secret),
    )
    return response.json()["access_token"]

token = get_access_token(f"https://{server}:{port}/connect/token", clientId, secret)

client = ApiClient(
    configuration=Configuration(host=f'https://{server}:{port}/api/ebay/v1'), header_name='Authorization',
    header_value='Bearer ' + token)

api = DefaultApi(client)

In [94]:
currencies = api.get_currencies()
currency_rates = {}
for currency in currencies:
    if (datetime.now() - datetime.strptime(currencies[0].last_update, '%Y-%m-%dT%H:%M:%S.%fZ')).days > 1:
        raise Exception("exchange rate isn't accurate " + currency.ebay_name)
    currency_rates[currency.ebay_name] = currency.rate

productRowsExcluded = {'search_queries'}
lotRowsExcluded = {'seller', 'located_in', 'purchase_history'}
purchaseExcluded = {}

products: List[ProductWithId] = api.get_all_products()
df = pd.DataFrame()

for product in products:
    print(f'Processing {product.name}')
    lots: List[LotInfoShort] = api.get_lots(product_id=product.id)
    
    productRow = {}
    for key, value in product.__dict__.items():
        if key not in productRowsExcluded:
            productRow[f'product_{key}'] = value
        
    dataFrameArray = []
    for lot in lots:
        lotRow = productRow.copy()
        for key, value in lot.__dict__.items():
            if key not in lotRowsExcluded:
                lotRow[f'lot_{key}'] = value
                
        for purchase in lot.purchase_history:
            purchaseRow = lotRow.copy()
            for key, value in purchase.__dict__.items():
                if key not in purchaseExcluded:
                    purchaseRow[f'purchase_{key}'] = value
            dataFrameArray.append(purchaseRow)

    df = pd.concat([df, pd.DataFrame(dataFrameArray)], ignore_index=True)

Processing 2Ж27Л
Processing 6CC31 TESLA
Processing 6Е1П
Processing 6Е3П
Processing 6И1П
Processing 6И1П-ЕВ
Processing 6Н15П
Processing 6Н16Б-В
Processing 6Н18Б-В
Processing 6Н1П
Processing 6Н1П-В
Processing 6Н1П-ВИ
Processing 6Н1П-Е
Processing 6Н1П-ЕВ
Processing 6Н1П-ЕВ СОВТЕК
Processing 6Н1П Совтек
Processing 6Н2П
Processing 6Н2П-В
Processing 6Н2П-Е
Processing 6Н2П-ЕВ
Processing 6Н2П-ЕР
Processing 6Н2П КИТАЙ
Processing 6Н3П
Processing 6Н3П-ДР
Processing 6Н3П-Е
Processing 6Н3П-ЕВ
Processing 6Н3П-И
Processing 6Н5П
Processing 6Н7С
Processing 6Н9С
Processing 6П13С
Processing 6П14П
Processing 6П14П-В
Processing 6П14П-ЕВ
Processing 6П14П-ЕР
Processing 6П14П-К
Processing 6П15П
Processing 6П15П-ЕВ
Processing 6П15П-ЕР
Processing 6П1П
Processing 6П1П-В
Processing 6П1П-Е
Processing 6П1П-ЕВ
Processing 6П21С
Processing 6П7С
Processing 6С19П
Processing 6С19П-В
Processing 6С19П-ВР
Processing 6С1П
Processing 6С2П
Processing 6С32Б
Processing 6С4П-Е
Processing 6С4П-ЕВ
Processing 6С51Н-В
Processing 6С52

In [95]:
df['purchase_price_filled_nulls'] = df.purchase_price.fillna(df.lot_price * unknown_price_discount)
df['exchange_rate'] = df.lot_currency.map(currency_rates)

In [96]:
df = df[(df.product_name == '6П14П')]

In [97]:
df['lot_manual_condition_id'].value_counts()

lot_manual_condition_id
usedAndMatched      246
newAndMatched        59
usedAndNotTested     46
usedAndTested        45
newAndTested         17
newNotTested         11
Name: count, dtype: int64

In [99]:
df['purchase_total_price'] = df.purchase_price_filled_nulls + df.lot_shipping + (df.lot_shipping_additional * (df.purchase_quantity - 1))

df['purchase_total_price_usd'] = df.purchase_total_price / df.exchange_rate

In [100]:
df

,product_id,product_name,product_last_check_time,lot_lot_id,lot_name,lot_pcs,lot_shipping_country,lot_currency,lot_price,lot_shipping,...,lot_condition,lot_condition_description,lot_manual_condition_id,purchase_price,purchase_quantity,purchase_var_date,purchase_price_filled_nulls,exchange_rate,purchase_total_price,purchase_total_price_usd
1100,a42a07b8-b1a1-465f-b625-edac46d65337,6П14П,2024-02-02T11:33:47.160Z,115043977132,10 PCS 6P14P / EL84 / 6BQ5 Vacuum Pentode Tube...,10,Germany,US $,19.0,8.95,...,Used,"Previously, vacuum tubes were used. Good condi...",usedAndNotTested,NaN,2,2024-01-29T11:27:01.000Z,13.30,1.0,31.20,31.20
1101,a42a07b8-b1a1-465f-b625-edac46d65337,6П14П,2024-02-02T11:33:47.160Z,115043977132,10 PCS 6P14P / EL84 / 6BQ5 Vacuum Pentode Tube...,10,Germany,US $,19.0,8.95,...,Used,"Previously, vacuum tubes were used. Good condi...",usedAndNotTested,NaN,3,2024-01-22T15:43:43.000Z,13.30,1.0,40.15,40.15
1102,a42a07b8-b1a1-465f-b625-edac46d65337,6П14П,2024-02-02T11:33:47.160Z,115043977132,10 PCS 6P14P / EL84 / 6BQ5 Vacuum Pentode Tube...,10,Germany,US $,19.0,8.95,...,Used,"Previously, vacuum tubes were used. Good condi...",usedAndNotTested,17.0,1,2024-01-19T17:33:53.000Z,17.00,1.0,25.95,25.95
1103,a42a07b8-b1a1-465f-b625-edac46d65337,6П14П,2024-02-02T11:33:47.160Z,115043977132,10 PCS 6P14P / EL84 / 6BQ5 Vacuum Pentode Tube...,10,Germany,US $,19.0,8.95,...,Used,"Previously, vacuum tubes were used. Good condi...",usedAndNotTested,17.0,1,2024-01-19T03:54:36.000Z,17.00,1.0,25.95,25.95
1104,a42a07b8-b1a1-465f-b625-edac46d65337,6П14П,2024-02-02T11:33:47.160Z,115043977132,10 PCS 6P14P / EL84 / 6BQ5 Vacuum Pentode Tube...,10,Germany,US $,19.0,8.95,...,Used,"Previously, vacuum tubes were used. Good condi...",usedAndNotTested,NaN,4,2023-12-28T15:46:47.000Z,13.30,1.0,49.10,49.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1519,a42a07b8-b1a1-465f-b625-edac46d65337,6П14П,2024-02-02T11:33:47.160Z,394134039609,6P14P 4pcs USED TESTED MATCHED QUAD REFLECTOR ...,4,Germany,US $,7.5,9.50,...,Used,USED / TESTED /SEE CONDITION AT THE PHOTO,usedAndMatched,7.5,1,2023-11-26T18:30:21.000Z,7.50,1.0,17.00,17.00
1520,a42a07b8-b1a1-465f-b625-edac46d65337,6П14П,2024-02-02T11:33:47.160Z,394134039609,6P14P 4pcs USED TESTED MATCHED QUAD REFLECTOR ...,4,Germany,US $,7.5,9.50,...,Used,USED / TESTED /SEE CONDITION AT THE PHOTO,usedAndMatched,NaN,4,2023-09-04T21:56:47.000Z,5.25,1.0,25.25,25.25
1521,a42a07b8-b1a1-465f-b625-edac46d65337,6П14П,2024-02-02T11:33:47.160Z,394134039609,6P14P 4pcs USED TESTED MATCHED QUAD REFLECTOR ...,4,Germany,US $,7.5,9.50,...,Used,USED / TESTED /SEE CONDITION AT THE PHOTO,usedAndMatched,NaN,1,2023-04-30T08:10:44.000Z,5.25,1.0,14.75,14.75
1522,a42a07b8-b1a1-465f-b625-edac46d65337,6П14П,2024-02-02T11:33:47.160Z,394134039609,6P14P 4pcs USED TESTED MATCHED QUAD REFLECTOR ...,4,Germany,US $,7.5,9.50,...,Used,USED / TESTED /SEE CONDITION AT THE PHOTO,usedAndMatched,NaN,4,2023-02-20T07:55:42.000Z,5.25,1.0,25.25,25.25
